In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, initializers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import time

# Load and transform MNIST data

In [2]:
# load data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
MAX_PIXEL = 255  # each MNIST pixel is an 8-bit int between 0 (black) & 255 (white)

# reshape from 2D pixel matrices to normalized 1D feature vectors to use in neural net
pixel_dimension = x_train.shape
x_train = x_train.reshape((len(x_train), x_train.shape[1]*x_train.shape[2])) / MAX_PIXEL
x_test  = x_test.reshape((len(x_test), x_test.shape[1]*x_test.shape[2])) / MAX_PIXEL

# one-hot encoding
y_train = to_categorical(y_train)
y_test  = to_categorical(y_test)

# 1. Setup neural network pipeline

In [3]:
# helper function to build and test models
def run_experiment(activation='relu', optimizer='adam', dropout=False, batchnorm=False, init='he_uniform'):
    model = models.Sequential()
    model.add(layers.Input(shape=(784,)))

    # hidden layer 1: learn wide range of abstract features with 128 neurons
    model.add(layers.Dense(128, activation=activation, kernel_initializer=init))
    if batchnorm: model.add(layers.BatchNormalization())
    if dropout: model.add(layers.Dropout(0.5))

    # hidden layer 2: refine features with 64 neurons
    model.add(layers.Dense(64, activation=activation, kernel_initializer=init))
    if batchnorm: model.add(layers.BatchNormalization())
    if dropout: model.add(layers.Dropout(0.5))

    # output layer: output 1 neuron per digit class (0-9)
    model.add(layers.Dense(10, activation='softmax'))

    # multi-class classification
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    start = time.time()
    history = model.fit(x_train, y_train, epochs=5, batch_size=128, verbose=0, validation_split=0.1)
    end = time.time()

    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    return acc, end - start

# 2. Experiment with three activation functions:
1. linear
2. sigmoid (nonlinear)
3. relu (nonlinear)

In [4]:
experiment = 'Activation Function'
activations = ['linear', 'sigmoid', 'relu']

activation_summary = pd.DataFrame(columns=[experiment, 'Accuracy', 'Execution Time (sec)'])
for act in activations:
    acc, t = run_experiment(activation=act)
    activation_summary.loc[len(activation_summary)] = [act, acc, round(t, 2)]

activation_summary

,Activation Function,Accuracy,Execution Time (sec)
0,linear,0.9229,14.90
1,sigmoid,0.9598,14.39
2,relu,0.9759,13.90


# 3. Experiment with three optimizers:
1. sgd
2. adam
3. rmsprop

In [5]:
experiment = 'Optimizers'
optimizers_list = ['sgd', 'adam', 'rmsprop']

optimizer_summary = pd.DataFrame(columns=[experiment, 'Accuracy', 'Execution Time (sec)'])
for opt in optimizers_list:
    acc, t = run_experiment(optimizer=opt)
    optimizer_summary.loc[len(optimizer_summary)] = [opt, acc, round(t, 2)]

optimizer_summary

,Optimizers,Accuracy,Execution Time (sec)
0,sgd,0.9208,11.70
1,adam,0.9718,14.16
2,rmsprop,0.9747,16.14


# 4. Experiment with regularization:
1. “Dropout layer”
2. “BatchNorm”
3. “Weight initialization”

In [7]:
experiment = 'Regularization'

configs = [
    {'dropout': True},
    {'batchnorm': True},
    {'init': 'glorot_uniform'},
    {'init': 'random_normal'},
]

regularization_summary = pd.DataFrame(columns=[experiment, 'Accuracy', 'Execution Time (sec)'])
for cfg in configs:
    acc, t = run_experiment(**cfg)
    regularization_summary.loc[len(regularization_summary)] = [cfg, acc, round(t, 2)]

regularization_summary

,Regularization,Accuracy,Execution Time (sec)
0,{'dropout': True},0.9617,20.33
1,{'batchnorm': True},0.9754,17.77
2,{'init': 'glorot_uniform'},0.9744,14.71
3,{'init': 'random_normal'},0.9717,16.52
